**Aktivitas Hands-on pertemuan 12**

Nama: Bagas Aditiya

NIM: 240401010141

Kelas: IF - 403

**Generate dan Eksplorasi Dataset**

In [3]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
np.random.seed(42)
produk = [
    'Roti', 'Selai', 'Susu', 'Sereal', 'Telur',
    'Keju', 'Kopi', 'Gula', 'Teh', 'Mentega'
]

transaksi = []

for _ in range(50):
    n_item = np.random.randint(2, 6)
    transaksi.append(
        list(np.random.choice(produk, n_item, replace=False))
    )

for i in range(20):
    if 'Roti' in transaksi[i] and 'Selai' not in transaksi[i]:
        transaksi[i].append('Selai')

print("Contoh transaksi:", transaksi[:3])
print("Jumlah transaksi:", len(transaksi))

Contoh transaksi: [[np.str_('Keju'), np.str_('Roti'), np.str_('Mentega'), np.str_('Kopi'), 'Selai'], [np.str_('Roti'), np.str_('Kopi'), np.str_('Teh'), np.str_('Selai'), np.str_('Mentega')], [np.str_('Kopi'), np.str_('Susu'), np.str_('Teh')]]
Jumlah transaksi: 50


Dataset sintetis berhasil dibuat sebanyak 50 transaksi dengan masing-masing transaksi terdiri dari 2–5 produk. Selain itu, ditambahkan pola tersembunyi yaitu produk Roti sering dibeli bersama Selai, sehingga nantinya algoritma Apriori diharapkan mampu menemukan hubungan tersebut.

**One-Hot Encoding Transaksi**

In [8]:
te = TransactionEncoder()
te_ary = te.fit(transaksi).transform(transaksi)
df = pd.DataFrame(te_ary, columns=te.columns_)
print(df.head())

    Gula   Keju   Kopi  Mentega   Roti  Selai  Sereal   Susu    Teh  Telur
0  False   True   True     True   True   True   False  False  False  False
1  False  False   True     True   True   True   False  False   True  False
2  False  False   True    False  False  False   False   True   True  False
3  False   True  False    False  False   True   False  False   True   True
4   True   True  False     True  False  False   False   True  False  False


Proses One-Hot Encoding berhasil mengubah daftar transaksi menjadi bentuk matriks biner sehingga setiap produk direpresentasikan oleh nilai True atau False. Bentuk data ini merupakan format yang dibutuhkan oleh algoritma Apriori.

**Pencarian Frequent Itemset dengan Apriori**

In [10]:
from mlxtend.frequent_patterns import apriori

for ms in [0.05, 0.1, 0.2]:
    freq = apriori(df, min_support=ms, use_colnames=True)
    print(f"min_support={ms}: {len(freq)} itemset ditemukan")

freq_items = apriori(df, min_support=0.1, use_colnames=True)
freq_items = freq_items.sort_values("support", ascending=False)

print(freq_items.head(10))

min_support=0.05: 74 itemset ditemukan
min_support=0.1: 44 itemset ditemukan
min_support=0.2: 13 itemset ditemukan
    support      itemsets
5      0.52       (Selai)
8      0.46         (Teh)
3      0.42     (Mentega)
9      0.36       (Telur)
1      0.34        (Keju)
0      0.32        (Gula)
2      0.32        (Kopi)
4      0.32        (Roti)
7      0.32        (Susu)
36     0.24  (Selai, Teh)


Semakin besar nilai minimum support, jumlah frequent itemset yang ditemukan semakin sedikit karena hanya kombinasi produk yang benar-benar sering muncul yang dipertahankan. Pada praktikum ini dipilih min_support = 0,1 karena menghasilkan jumlah itemset yang tidak terlalu sedikit maupun terlalu banyak sehingga lebih mudah dianalisis.

**Bentuk dan Saring Aturan Asosiasi**

In [11]:
rules = association_rules(freq_items, metric='confidence',
min_threshold=0.5)
rules = rules[rules['lift'] > 1].sort_values('lift', ascending=False)
print(rules[['antecedents', 'consequents',
'support', 'confidence', 'lift']].head(10))

         antecedents consequents  support  confidence      lift
8        (Keju, Teh)     (Telur)     0.12    0.857143  2.380952
15  (Mentega, Selai)      (Kopi)     0.10    0.625000  1.953125
12      (Gula, Roti)     (Selai)     0.10    1.000000  1.923077
7           (Sereal)   (Mentega)     0.14    0.777778  1.851852
10      (Teh, Telur)      (Keju)     0.12    0.600000  1.764706
14     (Kopi, Selai)   (Mentega)     0.10    0.714286  1.700680
9      (Keju, Telur)       (Teh)     0.12    0.750000  1.630435
11     (Selai, Gula)      (Roti)     0.10    0.500000  1.562500
13   (Kopi, Mentega)     (Selai)     0.10    0.714286  1.373626
1             (Roti)     (Selai)     0.22    0.687500  1.322115


Berdasarkan hasil Association Rules, aturan dengan nilai Lift tertinggi menunjukkan hubungan yang paling kuat antarproduk. Sebagai contoh, aturan Roti -> Selai memiliki nilai Lift yang tinggi, yang berarti pelanggan yang membeli roti memiliki kecenderungan lebih besar untuk juga membeli selai dibandingkan secara acak. Hasil ini masuk akal dari sisi bisnis karena kedua produk sering digunakan secara bersamaan.

**Membuat Rekomender Sederhana**

In [14]:
katalog = pd.DataFrame({
    'produk': produk,
    'kategori': [
        'Bakery', 'Bakery', 'Dairy', 'Bakery', 'Dairy',
        'Dairy', 'Minuman', 'Bumbu', 'Minuman', 'Dairy'
    ]
})
fitur = pd.get_dummies(katalog['kategori'])
sim_matrix = cosine_similarity(fitur)

def rekomendasi_serupa(nama_produk, top_n=3):
    idx = katalog.index[katalog['produk'] == nama_produk][0]
    skor = list(enumerate(sim_matrix[idx]))
    skor = sorted(
        skor,
        key=lambda x: x[1],
        reverse=True
    )
    skor = [s for s in skor if s[0] != idx][:top_n]
    return katalog.iloc[[i for i, _ in skor]]['produk'].tolist()
print("Mirip dengan Roti:", rekomendasi_serupa("Roti"))

Mirip dengan Roti: ['Selai', 'Sereal', 'Susu']


Content-Based Filtering memberikan rekomendasi berdasarkan kemiripan karakteristik produk, dalam hal ini kategori produk. Produk yang memiliki kategori yang sama dengan Roti memperoleh nilai kemiripan yang lebih tinggi sehingga direkomendasikan sebagai alternatif atau pelengkap.

**Membandingkan Kedua Pendekatan**

In [15]:
produk_target = 'Roti'
rules_terkait = rules[rules['antecedents'].apply(
lambda x: produk_target in x)]
print('Rekomendasi dari Association Rules:')
print(rules_terkait[['consequents', 'lift']].head())
print('Rekomendasi dari Content-Based:', rekomendasi_serupa(produk_target))

Rekomendasi dari Association Rules:
   consequents      lift
12     (Selai)  1.923077
1      (Selai)  1.322115
Rekomendasi dari Content-Based: ['Selai', 'Sereal', 'Susu']


Association Rules memberikan rekomendasi berdasarkan pola pembelian pelanggan sehingga mampu menemukan produk yang sering dibeli secara bersamaan. Sementara itu, Content-Based Filtering memberikan rekomendasi berdasarkan kemiripan karakteristik produk tanpa memperhatikan riwayat transaksi pelanggan. Pada praktikum ini kedua metode memberikan rekomendasi yang cukup konsisten karena sama-sama merekomendasikan Selai sebagai pasangan produk Roti. Dalam praktiknya, kedua pendekatan dapat digabungkan (Hybrid Recommendation) agar menghasilkan rekomendasi yang lebih akurat dan relevan.

**Kesimpulan:** Pada pertemuan ke-12 kali ini, saya mempelajari teknik Association Rule Mining menggunakan algoritma Apriori serta membangun sistem rekomendasi sederhana dengan pendekatan Content-Based Filtering. Proses yang dilakukan meliputi pembuatan dataset transaksi sintetis, pencarian frequent itemset, pembentukan aturan asosiasi menggunakan metrik support, confidence, dan lift, serta perbandingan hasil rekomendasi dengan metode berbasis kemiripan produk. Hasil praktikum menunjukkan bahwa algoritma Apriori mampu menemukan pola pembelian yang sering terjadi, seperti hubungan antara produk Roti dan Selai, sedangkan Content-Based Filtering merekomendasikan produk berdasarkan kesamaan kategori. Dari praktikum ini, saya memahami bahwa kedua metode memiliki keunggulan masing-masing dan dapat digabungkan untuk menghasilkan sistem rekomendasi yang lebih efektif.